1. Load all 10 stocks' unscaled technical data and count samples

In [1]:
import pandas as pd
import glob
import os

TECH_DIR = "data/technical"

def load_all_technical():
    frames = []
    for stock_dir in sorted(glob.glob(os.path.join(TECH_DIR, "*_NS"))):
        ticker = os.path.basename(stock_dir)
        path = os.path.join(stock_dir, "technical_featured_unscaled.csv")
        if not os.path.exists(path):
            print(f"[skip] {ticker}: no featured CSV found")
            continue
        df = pd.read_csv(path, parse_dates=["date"])
        df["ticker"] = ticker
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


all_tech = load_all_technical()
print("Total samples (all stocks, before split):", len(all_tech))
print("\nSamples per stock:")
print(all_tech.groupby("ticker").size())

Total samples (all stocks, before split): 11852

Samples per stock:
ticker
COFORGE_NS       1185
HCLTECH_NS       1186
INFY_NS          1186
LTM_NS           1183
LTTS_NS          1185
MPHASIS_NS       1185
PERSISTENT_NS    1185
TCS_NS           1186
TECHM_NS         1185
WIPRO_NS         1186
dtype: int64


In [2]:
FEATURE_COLUMNS = [
    "open", "high", "low", "close", "volume",
    "sma_10", "sma_20", "sma_50",
    "ema_12", "ema_26",
    "macd", "macd_signal", "macd_hist",
    "rsi_14",
    "bb_upper", "bb_lower", "bb_width",
    "atr_14",
    "obv", "adi",
    "stoch_k", "stoch_d",
    
    
]

summary = all_tech[FEATURE_COLUMNS].describe().T
summary["missing"] = all_tech[FEATURE_COLUMNS].isna().sum()
print(summary)

               count          mean           std           min           25%  \
open         11852.0  2.390557e+03  1.632396e+03  1.652000e+02  1.182874e+03   
high         11852.0  2.419204e+03  1.652287e+03  1.668186e+02  1.196258e+03   
low          11852.0  2.358115e+03  1.608742e+03  1.627724e+02  1.168865e+03   
close        11852.0  2.387922e+03  1.630109e+03  1.647146e+02  1.182787e+03   
volume       11852.0  3.421363e+06  5.865669e+06  8.405000e+03  4.460398e+05   
sma_10       11852.0  2.388202e+03  1.629779e+03  1.679932e+02  1.179263e+03   
sma_20       11852.0  2.388382e+03  1.629526e+03  1.689273e+02  1.184461e+03   
sma_50       11852.0  2.387655e+03  1.626650e+03  1.737082e+02  1.189049e+03   
ema_12       11852.0  2.388179e+03  1.628663e+03  1.693158e+02  1.181256e+03   
ema_26       11852.0  2.388134e+03  1.626529e+03  1.713732e+02  1.184217e+03   
macd         11852.0  4.522450e-02  6.551394e+01 -4.120618e+02 -2.191132e+01   
macd_signal  11852.0  1.578818e-01  6.18

In [4]:
import matplotlib.pyplot as plt

# per-stock sample counts
all_tech.groupby("ticker").size().plot(kind="bar", figsize=(10, 4), title="Samples per stock (before split)")
plt.ylabel("rows")
plt.tight_layout()
plt.savefig("technical_samples_per_stock.png")
plt.show()

# distribution of key features across all stocks combined
key_features = ["close", "volume", "rsi_14", "macd", "daily_return", "atr_14"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), key_features):
    ax.hist(all_tech[feat].dropna(), bins=50)
    ax.set_title(feat)
plt.tight_layout()
plt.savefig("technical_feature_distributions.png")
plt.show()

# close price scale per stock -- worth seeing since scaling is done per-stock
fig, ax = plt.subplots(figsize=(10, 5))
all_tech.boxplot(column="close", by="ticker", ax=ax, rot=45)
plt.title("Close price range per stock")
plt.suptitle("")
plt.tight_layout()
plt.savefig("technical_close_price_by_stock.png")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'